# LIMPIEZA Y CLASIFICACIÓN — NETFLIX CUSTOMER CHURN (versión resumida)

**Objetivo:** predecir si un cliente cancelará su suscripción (`churned`).

**Modelo: Regresión Logística.** La variable a predecir solo vale **0** (se queda) o **1** (se va), es decir es **binaria**, por lo que corresponde un modelo de **clasificación** y no una regresión lineal.

**Pasos:** cargar → explorar → limpiar → transformar → entrenar → evaluar → comparar con otros modelos.

## 1. Librerías y carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

# Se sube el archivo netflix_customer_churn.csv desde la computadora.
from google.colab import files
files.upload()

dataset = pd.read_csv('netflix_customer_churn.csv')
dataset.head()

ModuleNotFoundError: No module named 'google'

## 2. Exploración

In [ ]:
print('Dimensiones:', dataset.shape)
print('Valores nulos:', dataset.isnull().sum().sum())
print('Filas duplicadas:', dataset.duplicated().sum())
print()
print('Distribución de la variable objetivo (churned):')
print(dataset['churned'].value_counts(normalize=True).round(3))

Las clases están **balanceadas** (≈50% y ≈50%), por lo que la exactitud será una métrica confiable.

## 3. Limpieza de datos

Se eliminan las columnas sin utilidad, los nulos, los duplicados y los valores atípicos.

In [ ]:
# Se eliminan 3 columnas:
#   customer_id            → identificador, no predice nada.
#   monthly_fee            → redundante, depende 1 a 1 de subscription_type.
#   avg_watch_time_per_day → colineal con watch_hours (miden lo mismo).
dataset = dataset.drop(columns=['customer_id', 'monthly_fee', 'avg_watch_time_per_day'])

# Se eliminan nulos y duplicados.
dataset = dataset.dropna().drop_duplicates().reset_index(drop=True)

# Se guarda una copia para poder comparar el antes y el después.
antes = dataset.copy()

# Se eliminan valores atípicos con el rango intercuartílico (IQR).
for col in ['age', 'watch_hours', 'last_login_days', 'number_of_profiles']:
    q1, q3 = dataset[col].quantile(0.25), dataset[col].quantile(0.75)
    iqr = q3 - q1
    dataset = dataset[(dataset[col] > q1 - 1.5*iqr) & (dataset[col] < q3 + 1.5*iqr)]

# Se separan las filas que fueron descartadas.
atipicos = antes.drop(dataset.index)
dataset = dataset.reset_index(drop=True)

print('Datos originales :', antes.shape[0], 'clientes')
print('Atípicos elimin. :', atipicos.shape[0], 'clientes')
print('Datos finales    :', dataset.shape[0], 'clientes')

La siguiente gráfica muestra el proceso: los datos completos, los puntos que se descartan por atípicos (en rojo) y el conjunto final con el que se va a trabajar.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True, sharex=True)

# 1. Todos los datos.
ejes[0].scatter(antes['last_login_days'], antes['watch_hours'], alpha=0.4)
ejes[0].set_title('1. Datos originales (' + str(len(antes)) + ')')

# 2. Los mismos datos, señalando en rojo los que se van a eliminar.
ejes[1].scatter(antes['last_login_days'], antes['watch_hours'], alpha=0.4)
ejes[1].scatter(atipicos['last_login_days'], atipicos['watch_hours'], color='red', alpha=0.7)
ejes[1].set_title('2. Atípicos detectados (' + str(len(atipicos)) + ')')

# 3. El conjunto final, ya sin los atípicos.
ejes[2].scatter(dataset['last_login_days'], dataset['watch_hours'], alpha=0.4, color='green')
ejes[2].set_title('3. Datos con los que se trabaja (' + str(len(dataset)) + ')')

for eje in ejes:
    eje.set_xlabel('Días desde el último acceso')
ejes[0].set_ylabel('Horas vistas')

plt.tight_layout()
plt.show()

## 4. Transformación

El modelo solo acepta números, y en escalas comparables.

In [ ]:
# Se convierten las columnas de texto en números.
le = LabelEncoder()
columnas_texto = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']
for col in columnas_texto:
    dataset[col] = le.fit_transform(dataset[col])

# Se separa la variable objetivo (no se escala, debe seguir valiendo 0 o 1).
y = dataset['churned']
X = dataset.drop(columns=['churned'])

# Se escalan las características al rango 0-1.
X = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

# Se guarda el conjunto ya preprocesado.
pd.concat([X, y], axis=1).to_csv('netflix_churn_limpio.csv', index=False)
X.head()

## 5. Entrenamiento del modelo

In [ ]:
# 80% entrenamiento, 20% prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]
print('Modelo entrenado.')

## 6. Evaluación

In [ ]:
print('Exactitud    :', round(accuracy_score(y_test, y_pred), 4))
print('Precisión    :', round(precision_score(y_test, y_pred), 4))
print('Sensibilidad :', round(recall_score(y_test, y_pred), 4))
print('Puntaje F1   :', round(f1_score(y_test, y_pred), 4))
print('ROC-AUC      :', round(roc_auc_score(y_test, y_prob), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Permanece', 'Abandona']))

# Matriz de confusión.
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Permanece', 'Abandona'], yticklabels=['Permanece', 'Abandona'])
plt.title('Matriz de confusión')
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.show()

## 7. ¿Qué variables explican el abandono?

Ventaja de la regresión logística: sus **coeficientes** indican cómo influye cada variable.
Positivo = aumenta la probabilidad de abandono. Negativo = la reduce.

In [ ]:
coef = pd.Series(modelo.coef_[0], index=X.columns).sort_values()

coef.plot(kind='barh', figsize=(8, 5),
          color=['#4C72B0' if v < 0 else '#C44E52' for v in coef])
plt.axvline(0, color='black', linewidth=1)
plt.title('Influencia de cada variable en el abandono')
plt.xlabel('Coeficiente')
plt.show()

print(coef.sort_values(ascending=False).round(3))

## 8. Comparación con otros modelos de clasificación

Se prueban otros modelos vistos en clase sobre el mismo conjunto de datos, evaluados con **cinco métricas** cada uno.

**Modelos comparados:** Regresión Logística, Árbol de Decisión, Random Forest y Máquina de Soporte Vectorial (SVM).

> Los modelos de **regresión lineal, múltiple y polinomial** no se incluyen porque predicen valores continuos, y aquí la variable objetivo es binaria (0 o 1). Son modelos de regresión, no de clasificación.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Se definen los modelos a comparar.
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

# Se entrena cada uno y se calculan sus métricas.
resultados = []
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(X_test)
    resultados.append({
        'Modelo': nombre,
        'Exactitud': accuracy_score(y_test, pred),
        'Precisión': precision_score(y_test, pred),
        'Sensibilidad': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })

tabla = pd.DataFrame(resultados).set_index('Modelo').round(4)
tabla

In [ ]:
# Se grafica la comparación.
tabla.plot(kind='bar', figsize=(11, 5), width=0.8)
plt.title('Comparación de modelos de clasificación')
plt.ylabel('Puntaje')
plt.ylim(0, 1.25)
plt.xticks(rotation=10)
plt.legend(loc='upper center', ncol=5, fontsize=9)
plt.grid(axis='y', alpha=0.3)
plt.show()

print('Mejor modelo según F1     :', tabla['F1'].idxmax(), round(tabla['F1'].max(), 4))
print('Mejor modelo según ROC-AUC:', tabla['ROC-AUC'].idxmax(), round(tabla['ROC-AUC'].max(), 4))

In [ ]:
# Matriz de confusión de cada modelo.
fig, ejes = plt.subplots(1, 4, figsize=(18, 4))

for eje, (nombre, m) in zip(ejes, modelos.items()):
    sns.heatmap(confusion_matrix(y_test, m.predict(X_test)), annot=True, fmt='d',
                cmap='Blues', cbar=False, ax=eje,
                xticklabels=['Perm.', 'Aband.'], yticklabels=['Perm.', 'Aband.'])
    eje.set_title(nombre)
    eje.set_xlabel('Predicción')

ejes[0].set_ylabel('Valor real')
plt.tight_layout()
plt.show()

## Conclusión

Se limpió el conjunto de datos (columnas inútiles, nulos, duplicados y atípicos), se transformó a formato numérico escalado y se entrenó una **regresión logística**, el modelo adecuado porque la variable objetivo es **binaria**.

Además de predecir quién abandonará, el modelo **explica por qué**: las horas vistas reducen la probabilidad de abandono y los días sin conectarse la aumentan.

Finalmente se compararon **cuatro modelos de clasificación** (Regresión Logística, Árbol de Decisión, Random Forest y SVM) con cinco métricas cada uno, para verificar cuál ofrece el mejor desempeño sobre este conjunto de datos.
